In [ ]:
import os

# See what's actually inside your Drive folder
print(os.listdir("/content/drive/MyDrive/SMA-NLP Project/"))

['Claude Chat-1.pdf', 'Claude Chat-2.pdf', 'depression_pre_features_tfidf_256.csv', 'ptsd_pre_features_tfidf_256.csv', 'jokes_pre_features_tfidf_256.csv', 'socialanxiety_pre_features_tfidf_256.csv', 'personalfinance_pre_features_tfidf_256.csv', 'teaching_pre_features_tfidf_256.csv', 'anxiety_pre_features_tfidf_256.csv', 'merged_raw.csv', 'merged_balanced.csv', 'train_df.csv', 'test_df.csv', 'model_unscrubbed.pkl', 'model_scrubbed.pkl']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

base_path = "/content/drive/MyDrive/SMA-NLP Project/"

expected_mb = {
    "depression_pre_features_tfidf_256.csv": 60.5,
    "anxiety_pre_features_tfidf_256.csv": 56.7,
    "socialanxiety_pre_features_tfidf_256.csv": 34.2,
    "ptsd_pre_features_tfidf_256.csv": 14.0,
    "jokes_pre_features_tfidf_256.csv": 32.9,
    "personalfinance_pre_features_tfidf_256.csv": 63.5,
    "teaching_pre_features_tfidf_256.csv": 6.1,
}

for fname, expected in expected_mb.items():
    full_path = base_path + fname
    if os.path.exists(full_path):
        size_mb = os.path.getsize(full_path) / (1024*1024)
        flag = "  <-- MUCH SMALLER THAN EXPECTED" if size_mb < expected * 0.8 else "  OK"
        print(f"{fname}: {size_mb:.1f} MB (expected ~{expected} MB){flag}")
    else:
        print(f"{fname}: FILE NOT FOUND")

depression_pre_features_tfidf_256.csv: 57.7 MB (expected ~60.5 MB)  OK
anxiety_pre_features_tfidf_256.csv: 54.1 MB (expected ~56.7 MB)  OK
socialanxiety_pre_features_tfidf_256.csv: 32.6 MB (expected ~34.2 MB)  OK
ptsd_pre_features_tfidf_256.csv: 13.4 MB (expected ~14.0 MB)  OK
jokes_pre_features_tfidf_256.csv: 31.3 MB (expected ~32.9 MB)  OK
personalfinance_pre_features_tfidf_256.csv: 60.5 MB (expected ~63.5 MB)  OK
teaching_pre_features_tfidf_256.csv: 5.8 MB (expected ~6.1 MB)  OK


In [ ]:
import pandas as pd

base_path = "/content/drive/MyDrive/SMA-NLP Project/"
KEEP = ["subreddit", "author", "date", "post"]  # confirm these match your actual columns

test_file = base_path + "depression_pre_features_tfidf_256.csv"

# Test with the C engine (default, faster, stricter about malformed quotes)
d_c = pd.read_csv(test_file, usecols=lambda c: c in KEEP, engine="c", on_bad_lines="skip")
print(f"C engine:      {len(d_c)} rows")

# Test with the Python engine (what we used before, slower, more tolerant)
d_py = pd.read_csv(test_file, usecols=lambda c: c in KEEP, engine="python", on_bad_lines="warn")
print(f"Python engine: {len(d_py)} rows")

C engine:      21209 rows
Python engine: 21209 rows


In [ ]:
import pandas as pd

base_path = "/content/drive/MyDrive/SMA-NLP Project/"
KEEP = ["subreddit", "author", "date", "post"]  # confirm these match your actual columns

files = {
    base_path + "depression_pre_features_tfidf_256.csv": "depression",
    base_path + "anxiety_pre_features_tfidf_256.csv": "anxiety",
    base_path + "ptsd_pre_features_tfidf_256.csv": "ptsd",
    base_path + "socialanxiety_pre_features_tfidf_256.csv": "social_anxiety",
    base_path + "jokes_pre_features_tfidf_256.csv": "control",
    base_path + "personalfinance_pre_features_tfidf_256.csv": "control",
    base_path + "teaching_pre_features_tfidf_256.csv": "control",
}

dfs = []
for f, label in files.items():
    d = pd.read_csv(f, usecols=lambda c: c in KEEP, engine="c", on_bad_lines="skip")
    assert set(KEEP).issubset(d.columns), f"schema mismatch in {f}"
    d["label"] = label
    dfs.append(d)
    print(f"Loaded {f.split('/')[-1]}: {len(d)} rows")

data = pd.concat(dfs, ignore_index=True)
print(f"\nTotal after merge: {len(data)}")
print(data.label.value_counts())

Loaded depression_pre_features_tfidf_256.csv: 21209 rows
Loaded anxiety_pre_features_tfidf_256.csv: 19976 rows
Loaded ptsd_pre_features_tfidf_256.csv: 4483 rows
Loaded socialanxiety_pre_features_tfidf_256.csv: 12277 rows
Loaded jokes_pre_features_tfidf_256.csv: 19249 rows
Loaded personalfinance_pre_features_tfidf_256.csv: 23440 rows
Loaded teaching_pre_features_tfidf_256.csv: 2350 rows

Total after merge: 102984
label
control           45039
depression        21209
anxiety           19976
social_anxiety    12277
ptsd               4483
Name: count, dtype: int64


In [ ]:
data.to_csv(base_path + "merged_raw.csv", index=False)
print("Saved merged_raw.csv to Drive.")

Saved merged_raw.csv to Drive.


In [ ]:
overlap = data.groupby("author")["label"].nunique()
leaky_users = overlap[overlap > 1]
print(f"{len(leaky_users)} users posted in more than one class — removing them")

data = data[~data["author"].isin(leaky_users.index)]
print(len(data), data.label.value_counts())

3359 users posted in more than one class — removing them
96003 label
control           44361
depression        19210
anxiety           17701
social_anxiety    10688
ptsd               4043
Name: count, dtype: int64


In [ ]:
control_target = 25000

control_data = data[data.label == "control"]
non_control_data = data[data.label != "control"]

control_sampled = (
    control_data
    .groupby("subreddit", group_keys=False)[control_data.columns]
    .apply(lambda x: x.sample(
        n=min(len(x), control_target // control_data.subreddit.nunique()),
        random_state=42
    ), include_groups=False)
)

# include_groups=False drops the subreddit column from the result, so we add it back
if "subreddit" not in control_sampled.columns:
    control_sampled["subreddit"] = control_data.loc[control_sampled.index, "subreddit"]

data_balanced = pd.concat([non_control_data, control_sampled], ignore_index=True)
print(len(data_balanced))
print(data_balanced.label.value_counts())

70583
label
depression        19210
control           18941
anxiety           17701
social_anxiety    10688
ptsd               4043
Name: count, dtype: int64


In [ ]:
data_balanced.to_csv(base_path + "merged_balanced.csv", index=False)
print("Saved merged_balanced.csv to Drive.")

Saved merged_balanced.csv to Drive.


In [ ]:
from sklearn.model_selection import train_test_split

authors = data_balanced["author"].unique()
train_authors, test_authors = train_test_split(authors, test_size=0.2, random_state=42)

train_df = data_balanced[data_balanced.author.isin(train_authors)].copy()
test_df = data_balanced[data_balanced.author.isin(test_authors)].copy()

print(f"Train: {len(train_df)}")
print(train_df.label.value_counts())
print(f"\nTest: {len(test_df)}")
print(test_df.label.value_counts())

Train: 56465
label
depression        15412
control           15168
anxiety           14100
social_anxiety     8539
ptsd               3246
Name: count, dtype: int64

Test: 14118
label
depression        3798
control           3773
anxiety           3601
social_anxiety    2149
ptsd               797
Name: count, dtype: int64


In [ ]:
import re

ARTIFACT_TERMS = [
    "depression", "anxiety", "ptsd", "social anxiety", "socialanxiety",
    "diagnosed", "diagnosis", "dae", "throwaway", "tw:", "trigger warning",
    "r/depression", "r/anxiety", "r/ptsd", "r/socialanxiety",
]
# Short-form self-references — matched with word boundaries so we don't
# accidentally strip real words like "sad" or "gadget"
ARTIFACT_TERMS_WB = ["sa", "gad"]

def scrub_text(text):
    text = str(text).lower()
    for term in ARTIFACT_TERMS:
        text = re.sub(re.escape(term), "", text)
    for term in ARTIFACT_TERMS_WB:
        text = re.sub(r"\b" + re.escape(term) + r"\b", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["text_unscrubbed"] = train_df["post"]
train_df["text_scrubbed"] = train_df["post"].apply(scrub_text)

test_df["text_unscrubbed"] = test_df["post"]
test_df["text_scrubbed"] = test_df["post"].apply(scrub_text)

train_df.to_csv(base_path + "train_df.csv", index=False)
test_df.to_csv(base_path + "test_df.csv", index=False)

print("Done. train_df and test_df saved to Drive.")

Done. train_df and test_df saved to Drive.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight
import time

le = LabelEncoder()
y_train = le.fit_transform(train_df["label"])
y_test = le.transform(test_df["label"])

t0 = time.time()
vectorizer_unscrubbed = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.9,
    sublinear_tf=True
)
X_train_unscrubbed = vectorizer_unscrubbed.fit_transform(train_df["text_unscrubbed"])
X_test_unscrubbed = vectorizer_unscrubbed.transform(test_df["text_unscrubbed"])
print(f"Vectorized in {time.time()-t0:.1f}s | shape: {X_train_unscrubbed.shape}")

t1 = time.time()
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

model_unscrubbed = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.15,
    tree_method="hist",
    objective="multi:softprob",
    num_class=len(le.classes_),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)
model_unscrubbed.fit(X_train_unscrubbed, y_train, sample_weight=sample_weights)
print(f"Trained in {time.time()-t1:.1f}s")

preds_unscrubbed = model_unscrubbed.predict(X_test_unscrubbed)
print(classification_report(y_test, preds_unscrubbed, target_names=le.classes_, digits=3))

Vectorized in 19.8s | shape: (56465, 10000)
Trained in 631.2s
                precision    recall  f1-score   support

       anxiety      0.832     0.734     0.780      3601
       control      0.881     0.969     0.923      3773
    depression      0.831     0.821     0.826      3798
          ptsd      0.777     0.801     0.789       797
social_anxiety      0.697     0.720     0.708      2149

      accuracy                          0.822     14118
     macro avg      0.804     0.809     0.805     14118
  weighted avg      0.821     0.822     0.820     14118



In [ ]:
import pickle

with open(base_path + "model_unscrubbed.pkl", "wb") as f:
    pickle.dump({"model": model_unscrubbed, "vectorizer": vectorizer_unscrubbed, "label_encoder": le}, f)

print("Saved model_unscrubbed.pkl to Drive.")

Saved model_unscrubbed.pkl to Drive.


In [ ]:
t0 = time.time()
vectorizer_scrubbed = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.9,
    sublinear_tf=True
)
X_train_scrubbed = vectorizer_scrubbed.fit_transform(train_df["text_scrubbed"])
X_test_scrubbed = vectorizer_scrubbed.transform(test_df["text_scrubbed"])
print(f"Vectorized in {time.time()-t0:.1f}s | shape: {X_train_scrubbed.shape}")

t1 = time.time()
model_scrubbed = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.15,
    tree_method="hist",
    objective="multi:softprob",
    num_class=len(le.classes_),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)
model_scrubbed.fit(X_train_scrubbed, y_train, sample_weight=sample_weights)
print(f"Trained in {time.time()-t1:.1f}s")

preds_scrubbed = model_scrubbed.predict(X_test_scrubbed)
print(classification_report(y_test, preds_scrubbed, target_names=le.classes_, digits=3))

Vectorized in 10.9s | shape: (56465, 10000)
Trained in 643.2s
                precision    recall  f1-score   support

       anxiety      0.740     0.634     0.683      3601
       control      0.871     0.958     0.912      3773
    depression      0.778     0.751     0.764      3798
          ptsd      0.596     0.675     0.633       797
social_anxiety      0.656     0.709     0.682      2149

      accuracy                          0.766     14118
     macro avg      0.728     0.745     0.735     14118
  weighted avg      0.765     0.766     0.763     14118



In [ ]:
import pickle

with open(base_path + "model_scrubbed.pkl", "wb") as f:
    pickle.dump({"model": model_scrubbed, "vectorizer": vectorizer_scrubbed, "label_encoder": le}, f)

print("Saved model_scrubbed.pkl to Drive.")

Saved model_scrubbed.pkl to Drive.
